# KSS 30 分钟上手：横截面预测 + Walk-forward 回测

**新人看这一个 notebook 就够了**——读完后你应该能：

1. 用 `kss.data` / `kss.features` 把原始 CSV 转成多股票面板，自动生成 49+ 因子；
2. 用 `CrossSectionalForecast` 做「今天该买谁」的截面 ranking 选股；
3. 跑 `BacktestEngine.walk_forward`，并启用 Wave 1-3 的三大新能力——`neutralize`（行业/市值中性化）、`model_type='ranker'`（lambdarank 替代 MSE）、`ExecutionModel`（涨停过滤 + 部分成交 + 开盘冲击）；
4. 跑 `Significance.sharpe_significance` 与 `StrategyRegistry.register` 这道**硬上线门槛**，理解为什么 KSS 体系内**唯一**通过门槛的策略是 `log_mv` 反向（科创板小市值因子）。

为什么是 `log_mv` 反向？仓库的 `docs/solutions/lookahead_bias_lessons.md` 总结了 7 轮实验：Sharpe 从单股票的 1.18 一路衰减到 LGB 多因子 walk-forward 的 -0.53，唯一活下来的真 alpha 是 `log_mv` 反向（Sharpe 1.93、p=0.017、DSR=0.754）。这条衰减曲线本身就是方法论。

In [ ]:
from __future__ import annotations

import glob
import logging
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

# kss 主要模块（注意演示了至少 5 个）
from kss.features.pipeline import FactorPipeline
from kss.data.industry_mapping import IndustryMapping
from kss.prediction.cross_sectional_forecast import CrossSectionalForecast
from kss.backtest.cost_model import CostModel, ExecutionModel
from kss.backtest.engine import BacktestEngine
from kss.backtest.cross_section import factor_cross_section_backtest
from kss.backtest.metrics import Metrics
from kss.backtest.significance import Significance
from kss.strategies.registry import StrategyRegistry, DeploymentBlockedError

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.WARNING)

# 仓库根：notebook 在 kss/notebooks/，向上回退两级
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'cs_data_688008.csv').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
print('REPO_ROOT =', REPO_ROOT)

## 1. 加载数据

**为什么这么做**：KSS 用 long panel（每行一只股票 × 一个交易日）做截面回测。`cs_data_688*.csv` 是 51 只科创板的日频行情快照（2023-01 ~ 2026 年初，约 805 天），是体系内默认的「中小池」测试集——比单股票样本量大、计算又快得过 A 股全市场。

`FactorPipeline` 会一口气产出 49+ 因子（技术 + 波动 + 量价 + 估值），并通过 `add_targets` 加 3 个训练标签（`future_return_5d/10d/20d`，close-to-close）和 1 个**回测期实盘可达**标签 `next_day_return`（`open[t+2] / open[t+1] - 1`，强制 T+1 开盘建仓避免 look-ahead）。

In [ ]:
files = sorted(glob.glob(str(REPO_ROOT / 'cs_data_688*.csv')))
print(f'找到 {len(files)} 只科创板股票')

all_panels: list[pd.DataFrame] = []
feature_cols: list[str] = []

for fp_path in files:
    df = pd.read_csv(fp_path)
    df['trade_date'] = pd.to_datetime(df['trade_date'])
    df = df.sort_values('trade_date').reset_index(drop=True)
    if len(df) < 100:
        continue
    fp = FactorPipeline(df)
    factors = fp.generate()
    factors = fp.add_targets(factors, df['close'])
    # ExecutionModel 需要的列也透传过来（pre_close / open / amount）
    for col in ('pre_close', 'open', 'amount'):
        if col in df.columns:
            factors[col] = df[col].values
    factors['symbol'] = df['ts_code'].iloc[0]
    all_panels.append(factors)
    if not feature_cols:
        feature_cols = fp.get_feature_cols(factors)

panel = pd.concat(all_panels, ignore_index=True)
# 补 industry 列（neutralize 用）：CSV 缺失则走 fallback_kcb
ind_map = IndustryMapping.fallback_kcb(panel['symbol'].unique())
panel = ind_map.add_to_panel(panel, symbol_col='symbol')

print(f'Panel: {len(panel):,} 行 × {panel["symbol"].nunique()} 股票 × {len(feature_cols)} 因子')
print(f'日期: {panel["trade_date"].min().date()} ~ {panel["trade_date"].max().date()}')
print(f'前 5 个因子: {feature_cols[:5]}')
panel[['symbol', 'trade_date', 'close', 'log_mv', 'industry', 'next_day_return']].head(3)

## 2. 横截面预测（推荐路径）

**为什么这么做**：`CrossSectionalForecast` 是 KSS 给「今天该买谁」问题的**首选答案**——它直接对单因子做截面 ranking，不训练模型，不做参数搜索。这是仓库经过 8 轮实验后得到的结论：在 A 股弱信号截面上，LGB MSE 预测+阈值切信号几乎必跑负，而单因子 rank 才是稳定 alpha 的来源。

`direction='long_low'` + `factor_col='log_mv'` = 选市值最小的 20%，等权持有。这就是 KSS 体系内**唯一通过上线门槛**的策略，业内俗称「小市值反向」。

In [ ]:
forecast = CrossSectionalForecast(
    factor_col='log_mv',
    direction='long_low',   # 小市值买入 → 等权持有
    top_pct=0.2,
    freshness_days=400,     # demo 数据可能不是「今天」，放宽陈旧度阈值
)

pool = forecast.predict_pool(panel)
print(f'截面日期: {pool.attrs.get("target_date").date()}')
print(f'全池 {len(pool)} 只，Top {(pool["in_top"]).sum()} 只入选')
pool[pool['in_top']].head(10)[['symbol', 'trade_date', 'factor_value', 'rank_position', 'planned_weight']]

## 3. 单股查询

**为什么这么做**：portfolio 视图回答「今天该买谁」，但运营经常被问「688322 现在怎么样？」——`predict_single` 就是给这种场景：返回该股票在当日截面的整数排名、百分位、verdict（`buy` / `hold` / `ignore`），可以直接接到飞书/钉钉日报。

In [ ]:
info = forecast.predict_single(panel, '688322.SH')
for k, v in info.items():
    print(f'  {k:18s} = {v}')
print('\n--- Markdown 输出 ---')
print(forecast.format_single_markdown(info))

## 4. 完整回测（带 Wave 1-3 三大新能力）

**为什么这么做**：`CrossSectionalForecast` 只回答「今天」，`walk_forward` 才告诉你「如果一直按这套规则跑过去 N 个月，曲线长什么样」。本 cell 同时打开 Wave 1-3 的三大能力（这些是 2026 年陆续加入 engine 的）：

- **`neutralize=True`**（Wave 1）：每个训练/测试窗口先把因子对行业 dummy + log_mv 跑 OLS，留残差。防止「以为找到 alpha，其实只是 size 因子 / 行业 β 暴露」。
- **`model_type='ranker'`**（Wave 2）：用 LambdaRank 替代 MSE 回归——弱 IC 信号下 ranker 显著优于 MSE，因为目标和评测对齐都是排序。
- **`execution=ExecutionModel(...)`**（Wave 3）：选 Top Pct 之后剔除买入侧涨停股（科创板 ±20%），按 `max_tradable_ratio` 按部分成交缩权重，并把开盘冲击成本叠到 `CostModel` 之上。

**⚠️ 速度**：51 只 × 805 天跑全样本 walk_forward 大约 3 分钟。这里只演示**机制**——用 panel 末尾 ~60 个交易日 + train_window=20 + retrain_freq=5，把循环压到 ~10-30 秒。要看真实业绩去第 6 节。

In [ ]:
# 截取末尾 ~60 个交易日，演示 walk_forward 机制（不追求统计意义上的 Sharpe）
all_dates = sorted(panel['trade_date'].unique())
demo_dates = set(all_dates[-60:])
demo_panel = panel[panel['trade_date'].isin(demo_dates)].copy()
print(f'Demo 切片: {len(demo_panel):,} 行 × {len(demo_dates)} 天')

engine = BacktestEngine(CostModel(buy_cost=0.001, sell_cost=0.002))
execution = ExecutionModel(
    limit_up_pct=0.10,
    kcb_limit_pct=0.20,        # 科创板 ±20%
    max_volume_pct=0.05,       # 单股不吃超 5% 成交额
    open_slippage_bps=10,      # 开盘冲击 10 bps
)

wf_result = engine.walk_forward(
    demo_panel,
    feature_cols=feature_cols,
    label_col='future_return_5d',
    train_window=20,             # 用 20 天训练
    retrain_freq=5,              # 5 天 retrain 一次 → 约 5-7 个测试窗口
    top_pct=0.2,
    min_train=200,
    min_test=20,
    min_stocks=8,
    # === Wave 1-3 ===
    neutralize=True,             # 行业 + log_mv 中性化
    model_type='ranker',         # LambdaRank 替代 MSE
    execution=execution,         # 涨停过滤 + 部分成交
)

if wf_result is None or wf_result.empty:
    print('⚠️ walk_forward 无有效结果（demo 数据量过小是正常的；本 cell 是机制演示）')
else:
    print(f'\n回测交易日数: {len(wf_result)}')
    print(wf_result[['trade_date', 'gross_return', 'net_return', 'turnover', 'n_stocks']].head())
    print('\n核心指标（demo 切片，仅看机制）：')
    m = Metrics.calc(wf_result['net_return'])
    for k in ('sharpe', 'annual', 'max_dd', 'win', 'n'):
        v = m.get(k, float('nan'))
        print(f'  {k:10s} = {v:.4f}' if isinstance(v, float) else f'  {k:10s} = {v}')

## 5. 显著性与上线门槛检查

**为什么这么做**：`Metrics.calc` 算 Sharpe，但**漂亮的 Sharpe 数字 ≠ 能上线**——`Significance.sharpe_significance` 同时给出 t-stat / p-value / Deflated Sharpe（DSR，扣除「跑了多少策略才选出这一个」的 selection bias），`StrategyRegistry.register` 把三道门槛固化成上线 gate：

1. Sharpe ≥ 0.5（默认）
2. 日均收益 t-test p-value < 0.05
3. DSR ≥ 0.4（n_trials 按 `strategy_family` 自动选）

我们先跑一个**会失败**的（demo 切片样本太小），再跑一个**会通过**的（第 6 节基于全样本 `log_mv` 反向回测，是仓库唯一过门槛的策略）。

In [ ]:
registry = StrategyRegistry()

# Case 1: 用 demo 切片注册（应该失败：样本太小 / Sharpe 不稳定）
if wf_result is not None and not wf_result.empty:
    sig = Significance.sharpe_significance(wf_result['net_return'], n_trials=2)
    print('显著性诊断（demo 切片）：')
    for k, v in sig.items():
        print(f'  {k:18s} = {v:.4f}' if isinstance(v, float) else f'  {k:18s} = {v}')

    print('\n尝试注册 demo 策略：')
    try:
        registry.register(
            'demo_wave123',
            wf_result['net_return'],
            strategy_family='single_factor',
        )
        print('  ✅ 通过（小概率）')
    except DeploymentBlockedError as exc:
        print(f'  ❌ 被拦截（预期）: failures = {exc.failures}')
else:
    print('（wf_result 为空，跳过 Case 1）')

## 6. 对比：log_mv 单因子 vs LGB 多因子

**为什么这么做**：第 4 节是机制演示——要看**真实业绩**就跑全样本单因子横截面。`factor_cross_section_backtest` 不训练模型、直接用因子值做 score，是 KSS 里**最简单也最强**的回测路径。

这一节同时演示：

- `log_mv` 反向（**预期 Sharpe ≈ 1.9，过门槛**）vs `pe_ttm` / `macd_hist` 等其它因子（基本过不去）；
- 用 `factor_cross_section_backtest(execution=...)` 也能挂涨停过滤（Wave 3），保持与 `walk_forward` 同口径。

全样本 51 × 805 跑这个大约 5-15 秒。

In [ ]:
cost = CostModel(buy_cost=0.001, sell_cost=0.002)
exec_full = ExecutionModel(kcb_limit_pct=0.20, max_volume_pct=0.05, open_slippage_bps=10)

results: dict[str, dict] = {}
for fc, direction in [('log_mv', 'long_low'), ('pe_ttm', 'long_low'), ('macd_hist', 'long_high')]:
    if fc not in panel.columns:
        print(f'  跳过 {fc}（panel 缺列）')
        continue
    sub = panel.dropna(subset=[fc, 'next_day_return'])
    res = factor_cross_section_backtest(
        sub,
        factor_col=fc,
        cost_model=cost,
        top_pct=0.2,
        direction=direction,
        min_stocks=10,
        execution=exec_full,
    )
    if res is None or res.empty:
        print(f'  {fc}: 无结果')
        continue
    m = Metrics.calc(res['net_return'])
    results[fc] = {'res': res, 'metrics': m, 'direction': direction}
    print(
        f'  {fc:10s} ({direction:9s}): '
        f'Sharpe={m["sharpe"]:+.2f}  annual={m["annual"]*100:+6.1f}%  '
        f'max_dd={m["max_dd"]*100:+6.1f}%  n={m["n"]}'
    )

**跑一次硬门槛**：用上面 `log_mv` 反向回测的净收益，调 `StrategyRegistry.register(strategy_family='prior')`——`'prior'` 这个 family tag 告诉门槛「这不是数据挖出来的，是先验信念因子（小市值在 A 股有几十年学术证据）」，所以 `n_trials=1`、DSR 不会被惩罚太重。这是 KSS 唯一通过门槛的策略。

In [ ]:
if 'log_mv' in results:
    log_mv_returns = results['log_mv']['res']['net_return']
    sig = Significance.sharpe_significance(log_mv_returns, n_trials=1)
    print('log_mv 反向 显著性：')
    for k, v in sig.items():
        print(f'  {k:18s} = {v:.4f}' if isinstance(v, float) else f'  {k:18s} = {v}')

    print('\n尝试注册（strategy_family="prior"）：')
    try:
        record = registry.register(
            'log_mv_reverse',
            log_mv_returns,
            strategy_family='prior',
            notes='科创板小市值因子，先验信念，n_trials=1',
        )
        print(f'  ✅ 通过！registered_at={record.registered_at}')
        print(f'     metrics={record.metrics}')
    except DeploymentBlockedError as exc:
        print(f'  ❌ 未通过: {exc.failures}')
        print(f'     metrics={exc.metrics}')

    print(f'\nRegistry 当前已注册策略: {registry.list_strategies()}')
else:
    print('（log_mv 回测未跑成功，跳过门槛检查）')

## 总结 + 下一步

你刚刚跑过的：

| Cell | 演示 | KSS 模块 |
|------|------|---------|
| 2-3 | 加载 + 因子生成 | `kss.features.pipeline.FactorPipeline` + `kss.data.industry_mapping` |
| 4 | 截面 ranking 选股 | `kss.prediction.CrossSectionalForecast` |
| 6 | 单股 verdict | `CrossSectionalForecast.predict_single` |
| 8 | walk-forward 三大新能力 | `kss.backtest.BacktestEngine.walk_forward(neutralize=True, model_type='ranker', execution=...)` |
| 10 | 显著性 + 上线门槛 | `kss.backtest.Significance.sharpe_significance` + `kss.strategies.StrategyRegistry.register` |
| 12 | 真实业绩对比 | `kss.backtest.cross_section.factor_cross_section_backtest` |
| 14 | log_mv 反向过门槛 | 唯一过 `is_deployable` 的策略 |

**关键认知**（来自 `docs/solutions/lookahead_bias_lessons.md`）：

1. 任何高 Sharpe 数字未经 7 层 bias（单股 / 阈值 / 静态权重 / 全样本选因子 / 行业暴露 / 实盘成交 / 多策略选择）逐一证伪前都是**虚高**——`StrategyRegistry` 把第 7 层固化成代码 gate。
2. A 股弱信号截面上，**单因子 ranking 通常比 LGB 多因子组合更稳**——这违反直觉但有大量实证。
3. `log_mv` 反向**不是被数据挖出来的**，是先验信念（小市值溢价有几十年学术证据），所以 `strategy_family='prior'` → `n_trials=1` → DSR 不被惩罚——这就是它能过门槛的原因。

**下一步可探索**：

- 把 `panel` 换成全 A 股（`kss/data/sqlite_store.py` 已支持）；
- 试 `CrossSectionalForecast(execution=...)` 加涨停过滤路径；
- 看 `kss/backtest/diagnostics.py` 里的 `SignalDiagnostics`（IC / 分位 / IC 衰减）；
- 看 `scripts/paper_trade_log_mv.py` 是怎么把这套截面预测接到实盘信号推送的。